# 3.1 Table One

**Core series, class 6.** Almost every health study starts with a **Table 1**: who is in each group, and how they differ. It tells readers what the groups look like, shows differences that later models must account for (like age), and reveals **missing data**. In this class we build a Table 1 for our cases and controls, writing our own small functions.

### By the end of this notebook you'll be able to

1. Explain what a Table 1 shows, and why every study has one
2. Summarise **categorical** variables (counts and percents) and **continuous** ones (mean and SD, or median and IQR)
3. Choose and read a **chi-square test** and a **t-test**
4. Write your own **function**, and run it on many variables with `map()`
5. Report **missing data**, and check for **small counts** before sharing

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | How do people with and without type 2 diabetes differ in age, sex, race, ethnicity, and measurements? |
| 🧪 **The analysis** | Counts and percents (chi-square tests) for categories; means and SDs (t-tests) for numbers; the percent with each measurement |
| 🎯 **Why** | To describe the groups, and to spot **confounders**: variables that differ between groups and also affect the outcome (age, most of all) |
| ✅ **What we conclude** | Where the groups differ, and how much data each variable has |
| ⚠️ **What it can't tell us** | **Why** they differ: a Table 1 is descriptive, unadjusted. And our sample is half cases, so percents describe the **sample**, not the population |
| 🔗 **Connections** | Uses `final_analysis.csv` from **2.3**; the differences it shows guide the models in **3.2–3.4** |

| | |
|---|---|
| 📂 **Uses** | `final_analysis.csv` (from **2.3**) |
| 💾 **Saves** | nothing (the table is shown in the notebook) |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Load | 5 |
| 3. Categorical variables | 20 |
| 4. Continuous variables | 15 |
| 5. Missing data, and the full table | 10 |
| 6. Small-count check | 5 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 75 minutes |
| ⏱️ **Run time** | under **1 minute** to run every cell |
| 💰 **Data cost** | **$0**: no BigQuery queries |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$0.35** per student for this class (app time only) |

> ⚠️ **Keep All of Us data in the Workbench.** Don't share counts of 1 to 20: Part 6 checks the table before you share it. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### What goes in a Table 1

| Variable type | Example | Summary | Test (are the groups different?) |
|---|---|---|---|
| **Categorical** | sex at birth, age group, race | **n (%)** in each group | **chi-square test** |
| **Continuous** | age, BMI, blood pressure | **mean (SD)**; or **median (IQR)** if skewed | **t-test** (or Wilcoxon for skewed data) |

- **SD** (standard deviation): how spread out the values are around the mean.
- **IQR** (interquartile range): the middle half of the values, from the 25th to the 75th percentile.
- **p-value:** how surprising the difference would be if the groups were really the same. With **thousands** of people, even tiny differences give small p-values, so look at the **size** of the difference, not just the p-value.

### Our sample is half cases

We sampled **5,000 cases and 5,000 controls**, but only about 1 in 5 adults in the cohort has type 2 diabetes. So "50% have diabetes" describes our **sample**, by design. Table 1 compares the two groups; it doesn't estimate how common anything is in the population.

### Functions: write once, use many times

We'll summarise many variables the same way, so we write a **function**: a named recipe that takes inputs and returns a result.

```r
add_one <- function(x) {
  x + 1
}
add_one(5)   # 6
```

| Code | Means |
|---|---|
| `function(data, var) { … }` | a function with two inputs |
| `data[[var]]` or `.data[[var]]` | "the column whose **name** is stored in `var`" |
| `map(vars, ~ f(data, .x))` + `list_rbind()` | run `f()` once per variable, and stack the results |

## Part 2. Load

`prepare_analysis_data()`, in `aou_setup.R`, reads `final_analysis.csv` and sets each column's type (e.g. `t2d` as a **factor** with *No* as the reference level).

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

fa <- prepare_analysis_data()
fa %>% count(t2d)

## Part 3. Categorical variables

`summarise_categorical()` makes the rows for one categorical variable:

1. counts each level in each group (people with `NA` for this variable are left out, and reported in Part 5),
2. turns the counts into **n (%)** within each group,
3. runs a **chi-square test** comparing the groups,
4. keeps the raw counts (`n_no`, `n_yes`) for the small-count check in Part 6.

In [ ]:
summarise_categorical <- function(data, var) {
  counts <- data %>%
    filter(!is.na(.data[[var]])) %>%
    count(level = as.character(.data[[var]]), t2d) %>%
    pivot_wider(names_from = t2d, values_from = n, values_fill = 0)

  p <- suppressWarnings(chisq.test(as.matrix(counts[, c("No", "Yes")]))$p.value)

  counts %>%
    mutate(`No diabetes` = sprintf("%d (%.1f%%)", No,  100 * No  / sum(No)),
           `Diabetes`    = sprintf("%d (%.1f%%)", Yes, 100 * Yes / sum(Yes)),
           variable = var,
           p_value  = c(format.pval(p, digits = 2, eps = 0.001), rep("", n() - 1))) %>%
    select(variable, level, `No diabetes`, `Diabetes`, p_value, n_no = No, n_yes = Yes)
}

categorical_vars <- c("sex_at_birth", "age_group", "race", "ethnicity")
table_cat <- map(categorical_vars, ~ summarise_categorical(fa, .x)) %>% list_rbind()

table_cat %>% select(-n_no, -n_yes)

**🔍 Read the output:** for each variable, one row per level, with **n (%)** in each group and one p-value per variable.

- `sprintf("%d (%.1f%%)", …)` builds text like `2,345 (46.9%)`: `%d` is a whole number, `%.1f` a number with 1 decimal, `%%` a percent sign.
- `.data[[var]]` lets one function work for any column name.
- **Age group:** cases are much more often 65+ (50% vs 34% in our test run) and rarely under 40 (5% vs 24%): age is the biggest difference, and must be in every model.
- **Race and ethnicity** are described here, as the cohort's make-up. They're not used to choose people, and differences between groups reflect many things (access to care, where people live, how diabetes is diagnosed), not biology.

### ✍️ Your turn

Add `self_reported_category` (*All of Us*'s combined race and ethnicity category) to the table.

<details>
<summary><b>Show answer</b></summary>

```r
summarise_categorical(fa, "self_reported_category") %>% select(-n_no, -n_yes)
```

It combines race and ethnicity into one variable, which is often easier to report than two. Check the small-count rule (Part 6) before sharing: smaller categories can have counts of 1 to 20.

</details>

In [ ]:
# Your code here

## Part 4. Continuous variables

`summarise_continuous()` gives **mean (SD)** in each group, the number of people with a value (`n`), and a **t-test**. `na.rm = TRUE` ignores missing values.

In [ ]:
summarise_continuous <- function(data, var) {
  x_no  <- data[[var]][data$t2d == "No"]
  x_yes <- data[[var]][data$t2d == "Yes"]
  tibble(
    variable      = var,
    level         = "mean (SD)",
    `No diabetes` = sprintf("%.1f (%.1f), n = %d", mean(x_no,  na.rm = TRUE), sd(x_no,  na.rm = TRUE), sum(!is.na(x_no))),
    `Diabetes`    = sprintf("%.1f (%.1f), n = %d", mean(x_yes, na.rm = TRUE), sd(x_yes, na.rm = TRUE), sum(!is.na(x_yes))),
    p_value       = format.pval(t.test(x_yes, x_no)$p.value, digits = 2, eps = 0.001)
  )
}

continuous_vars <- c("age", "BMI", "SBP", "LDL", "HbA1c")
table_cont <- map(continuous_vars, ~ summarise_continuous(fa, .x)) %>% list_rbind()
table_cont

**🔍 Read the output:**

- **Age:** cases are older, by several years on average.
- **BMI** is higher in cases, as expected: excess weight is a major risk factor for type 2 diabetes.
- **SBP** (systolic blood pressure) tends to be a little higher in cases.
- **LDL** is often **lower** in cases. That's **not** because diabetes protects against cholesterol: people with diabetes are usually prescribed **statins**, and our value is each person's **latest** LDL (2.2), mostly measured after treatment started.
- **HbA1c** is much higher in cases, **by definition**: it's one of the tests used to **diagnose** diabetes. It describes the groups, but can't be used to explain who has diabetes (3.2).
- `n` shows how many people had each measure: far fewer for LDL and HbA1c, and differently in each group.

**💬 Discuss:** if many controls never had an HbA1c test, who **are** the controls who did? (Often people whose doctor suspected diabetes: a selected group, which can make cases and controls look more alike.)

## Part 5. Missing data, and the full table

Readers need to know how complete each variable is, **in each group**:

In [ ]:
missing_rows <- map(c("sex_at_birth", "race", "ethnicity", continuous_vars),
                    ~ tibble(variable = .x, level = "% missing",
                             `No diabetes` = sprintf("%.1f%%", 100 * mean(is.na(fa[[.x]][fa$t2d == "No"]))),
                             `Diabetes`    = sprintf("%.1f%%", 100 * mean(is.na(fa[[.x]][fa$t2d == "Yes"]))),
                             p_value = "")) %>%
  list_rbind()

table_one <- bind_rows(
  tibble(variable = "N", level = "",
         `No diabetes` = as.character(sum(fa$t2d == "No")),
         `Diabetes`    = as.character(sum(fa$t2d == "Yes")), p_value = ""),
  table_cont,
  table_cat %>% select(-n_no, -n_yes),
  missing_rows
)

table_one %>% print(n = 60)

**🔍 Read the output:** your **Table 1**: group sizes, continuous variables, categorical variables, and missing data. In a paper, you'd also say how the sample was chosen and which values were kept (the cleaning rules from 2.1).

**HbA1c** and **LDL** are missing far more often in **controls**: the clearest sign that missingness here isn't random.

## Part 6. Small-count check

*All of Us* policy: **no counts of 1 to 20** in anything you share (including counts that could be worked out from others). Before sharing a table, list any cell that breaks the rule:

In [ ]:
table_cat %>%
  filter(between(n_no, 1, 20) | between(n_yes, 1, 20)) %>%
  select(variable, level, n_no, n_yes)

**🔍 Read the output:** ideally **no rows**. Any row listed **can't be shared** as it is. In our test run, four rows broke the rule: *Intersex* and *None of these* for sex at birth, and *Middle Eastern or North African* and *Native Hawaiian or Other Pacific Islander* for race.

### A shareable version

Two standard fixes, applied in this order:

1. **Combine** small levels into a broader one (e.g. *Another answer*, *Another race*). Combining loses detail, so say in a footnote which categories were combined.
2. Anything **still** between 1 and 20 is shown as **"20 or fewer"**.

`fct_other(x, keep = …)` (from **forcats**, part of the tidyverse) keeps the listed levels and puts every other level into one, named by `other_level`.

In [ ]:
fa_shared <- fa %>%
  mutate(sex_at_birth = fct_other(sex_at_birth, keep = c("Female", "Male"), other_level = "Another answer"),
         race = fct_other(race, keep = c("White", "Black or African American", "Asian",
                                         "American Indian or Alaska Native", "More than one population"),
                          other_level = "Another race"))

shareable <- map(c("sex_at_birth", "race"), ~ summarise_categorical(fa_shared, .x)) %>%
  list_rbind() %>%
  mutate(`No diabetes` = if_else(between(n_no, 1, 20),  "20 or fewer", `No diabetes`),
         `Diabetes`    = if_else(between(n_yes, 1, 20), "20 or fewer", `Diabetes`))

shareable %>% select(-n_no, -n_yes)

**🔍 Read the output:** the small categories are combined, and any count still from 1 to 20 shows as "20 or fewer". (The p-values now compare the combined categories.) Use tables like this one when you share results.

### ✍️ Your turn

BMI is **skewed** (a long tail of high values). Report BMI as **median (IQR)** instead of mean (SD).

<details>
<summary><b>Show answer</b></summary>

```r
fa %>%
  group_by(t2d) %>%
  summarise(median = median(BMI, na.rm = TRUE),
            q1 = quantile(BMI, 0.25, na.rm = TRUE),
            q3 = quantile(BMI, 0.75, na.rm = TRUE))

wilcox.test(BMI ~ t2d, data = fa)$p.value   # the matching test for skewed data
```

For skewed variables, the median and IQR describe a "typical" person better than the mean, and the **Wilcoxon** test replaces the t-test.

</details>

In [ ]:
# Your code here

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| *'final_analysis.csv' does not exist* | 2.3 hasn't been run | Run 2.1, 2.2, and 2.3 |
| *Chi-squared approximation may be incorrect* | Some expected counts are small | We hide it with `suppressWarnings()`; for small tables use `fisher.test()` |
| *not enough 'x' observations* in `t.test()` | One group has no values for that variable | Check `n` for that variable |
| A level shows `NA` | Missing values counted as a level | Our function filters `NA` first; check you're using it |

## ✅ Key takeaways

- A **Table 1** describes each group: n (%) for categories, mean (SD) or median (IQR) for numbers
- With large samples, judge the **size** of a difference, not just the p-value
- Report **missing data** by group: it's often not random
- **Functions** + `map()` turn one recipe into a whole table
- Check for **counts of 1 to 20** before sharing

**Next:** **3.2 logistic_regression**: which characteristics go with higher odds of type 2 diabetes, **adjusted** for each other?

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).